# What is a Neuron?

This notebook accompanies the **ML Viz** lesson on artificial neurons.
We'll implement a single neuron from scratch and visualize how it works.

**Companion lesson:** https://ml-viz.vercel.app/courses/neural-networks/01-what-is-a-neuron

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The neuron equation

$$y = f\left(\sum_{i=1}^n w_i x_i + b\right)$$

Three parts:
- **Inputs** $x_i$ — the data coming in
- **Weights** $w_i$ — how much each input matters
- **Bias** $b$ — shifts the activation threshold
- **Activation** $f$ — introduces non-linearity

In [ ]:
class Neuron:
    def __init__(self, weights, bias, activation='relu'):
        self.weights = np.array(weights, dtype=float)
        self.bias = float(bias)
        self.activation = activation

    def _activate(self, z):
        if self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        elif self.activation == 'tanh':
            return np.tanh(z)
        return z  # linear

    def forward(self, x):
        z = np.dot(self.weights, x) + self.bias
        return self._activate(z), z  # return (output, pre-activation)


# From the lesson exercise: weights=[2, -1, 0.5], bias=1, inputs=[1, 1, 1]
neuron = Neuron(weights=[2, -1, 0.5], bias=1, activation='relu')
output, z = neuron.forward([1, 1, 1])
print(f'Pre-activation z = {z}')   # Should be 2.5
print(f'Output (ReLU)  y = {output}')  # Should be 2.5

## Activation functions compared

Let's plot all four common activation functions side by side.

In [ ]:
x = np.linspace(-5, 5, 300)

activations = {
    'ReLU':    np.maximum(0, x),
    'Sigmoid': 1 / (1 + np.exp(-x)),
    'Tanh':    np.tanh(x),
    'Linear':  x,
}
colors = ['#818cf8', '#14b8a6', '#eab308', '#f97316']

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('Activation Functions', color='white', fontsize=13, y=1.02)

for ax, (name, y), color in zip(axes, activations.items(), colors):
    ax.plot(x, y, color=color, linewidth=2.5)
    ax.axhline(0, color='#2e3347', linewidth=0.8)
    ax.axvline(0, color='#2e3347', linewidth=0.8)
    ax.set_title(name, color='white', fontsize=11)
    ax.set_xlim(-5, 5)

plt.tight_layout()
plt.show()

## Why non-linearity matters

Without activation functions, stacking neurons is equivalent to a single linear transformation.
Let's prove this numerically.

In [ ]:
# Two-layer network without activation: W2 @ (W1 @ x + b1) + b2 = (W2@W1)@x + const
# This collapses to a single linear transformation.

np.random.seed(42)
W1 = np.random.randn(4, 2)   # 4 hidden neurons, 2 inputs
b1 = np.random.randn(4)
W2 = np.random.randn(1, 4)   # 1 output
b2 = np.random.randn(1)

x_test = np.array([3.0, -1.5])

# Without activation (linear)
h_linear = W1 @ x_test + b1          # hidden layer, no activation
y_linear = W2 @ h_linear + b2

# Equivalent single-layer
W_collapsed = W2 @ W1
b_collapsed = W2 @ b1 + b2
y_single = W_collapsed @ x_test + b_collapsed

print(f'Two-layer (no activation): {y_linear[0]:.6f}')
print(f'Collapsed single layer:    {y_single[0]:.6f}')
print(f'Difference (should be ~0): {abs(y_linear[0] - y_single[0]):.2e}')

# With ReLU — can no longer collapse
h_relu = np.maximum(0, W1 @ x_test + b1)
y_relu = W2 @ h_relu + b2
print(f'\nTwo-layer (with ReLU):     {y_relu[0]:.6f}  ← different!')

## What a single neuron can learn

A single neuron with sigmoid activation is equivalent to **logistic regression** — a linear decision boundary.

In [ ]:
# Generate linearly separable data
np.random.seed(7)
n = 80
X_pos = np.random.randn(n // 2, 2) + [1.5, 1.5]
X_neg = np.random.randn(n // 2, 2) + [-1.5, -1.5]
X = np.vstack([X_pos, X_neg])
y = np.array([1] * (n // 2) + [0] * (n // 2))

# Train a single sigmoid neuron via gradient descent
w = np.zeros(2)
b = 0.0
lr = 0.1

for _ in range(200):
    z = X @ w + b
    pred = 1 / (1 + np.exp(-z))
    err = pred - y
    w -= lr * X.T @ err / n
    b -= lr * err.mean()

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(*X_pos.T, color='#818cf8', s=30, label='Class 1', alpha=0.8)
ax.scatter(*X_neg.T, color='#f43f5e', s=30, label='Class 0', alpha=0.8)

# Decision boundary: w[0]*x + w[1]*y + b = 0  →  y = -(w[0]*x + b) / w[1]
xline = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
yline = -(w[0] * xline + b) / w[1]
ax.plot(xline, yline, '--', color='#14b8a6', linewidth=2, label='Decision boundary')
ax.legend()
ax.set_title('Single Neuron: Linear Decision Boundary', color='white')
plt.tight_layout()
plt.show()
print(f'Learned weights: {w.round(3)}, bias: {b:.3f}')

## A full neural network from scratch (2-layer MLP)

A single neuron only draws a straight line. Stack a hidden layer of neurons with a non-linear activation and you can carve **curved** boundaries. Here we build a 2-layer MLP, derive **backprop** by hand, and train it on a non-linearly-separable dataset.

In [ ]:
# Non-linearly-separable data: two concentric rings (inner=0, outer=1)
def make_rings(n=400, seed=0):
    rng = np.random.RandomState(seed)
    r = np.r_[rng.uniform(0, 1.0, n//2), rng.uniform(1.8, 2.8, n//2)]
    th = rng.uniform(0, 2*np.pi, n)
    X = np.c_[r*np.cos(th), r*np.sin(th)]
    y = np.r_[np.zeros(n//2), np.ones(n//2)].reshape(-1, 1)
    return X, y

X, y = make_rings()
plt.scatter(*X[y.ravel()==0].T, s=12, color='#818cf8', label='class 0')
plt.scatter(*X[y.ravel()==1].T, s=12, color='#f43f5e', label='class 1')
plt.legend(); plt.title('Not linearly separable — a single neuron cannot solve this'); plt.show()

### Forward pass, loss, and backprop

Two layers: $h=\tanh(XW_1+b_1)$, $\hat y=\sigma(hW_2+b_2)$, binary cross-entropy loss. Backprop is the chain rule: the output error $\hat y-y$ flows back through $W_2$, gets multiplied by the $\tanh'$ of the hidden layer, and updates $W_1$.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

class MLP:
    def __init__(self, n_in=2, n_hidden=16, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, n_hidden) * np.sqrt(2/n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.randn(n_hidden, 1) * np.sqrt(2/n_hidden)
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.h  = np.tanh(self.z1)
        self.p  = sigmoid(self.h @ self.W2 + self.b2)
        return self.p

    def backward(self, y, lr):
        n = y.shape[0]
        dz2 = (self.p - y) / n                 # dL/d(output pre-activation)
        dW2 = self.h.T @ dz2; db2 = dz2.sum(0, keepdims=True)
        dh  = dz2 @ self.W2.T
        dz1 = dh * (1 - self.h**2)             # times tanh'(z1)
        dW1 = self.X.T @ dz1; db1 = dz1.sum(0, keepdims=True)
        self.W2 -= lr*dW2; self.b2 -= lr*db2
        self.W1 -= lr*dW1; self.b1 -= lr*db1

def bce(p, y):
    return -np.mean(y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9))

net = MLP(n_hidden=16)
losses = []
for epoch in range(2000):
    p = net.forward(X)
    losses.append(bce(p, y))
    net.backward(y, lr=0.5)
acc = ((net.forward(X) > 0.5) == y).mean()
print(f'final loss {losses[-1]:.4f}   training accuracy {acc:.3f}')

### The learned non-linear boundary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(losses, color='#14b8a6'); ax[0].set_title('Training loss')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('BCE')

xx, yy = np.meshgrid(np.linspace(-3.2, 3.2, 250), np.linspace(-3.2, 3.2, 250))
Z = net.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[1].contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.7)
ax[1].scatter(*X[y.ravel()==0].T, s=10, color='#818cf8')
ax[1].scatter(*X[y.ravel()==1].T, s=10, color='#f43f5e')
ax[1].set_title('MLP decision boundary (a single neuron could never do this)')
plt.tight_layout(); plt.show()

## Key takeaways

- A neuron computes $y = f(\sum_i w_i x_i + b)$ — a weighted sum passed through an activation.
- **Weights** scale each input; the **bias** shifts the threshold.
- The **activation** $f$ adds non-linearity; without it, stacked layers collapse to one linear map.
- A single sigmoid neuron *is* logistic regression — a linear decision boundary.
- Depth + non-linearity is what lets networks model complex, hierarchical functions.